works in principle, now set up training loop

In [ ]:
import Pkg 
Pkg.activate("./..")

import QuantumGrav as QG
import Flux 
import CUDA 
import GraphNeuralNetworks as GNN
import CausalSets as CS
import Arrow
import Tables
import StatsBase 
import Statistics
import Random
import ProgressMeter
import MLUtils

In [ ]:
path_to_data = joinpath("/", "media", "hmack", "dataLinux", "machinelearning_data", "QuantumGrav", "unequal_size_data_small")
list_of_files = filter(x -> occursin(".arrow", x),readdir(path_to_data));

In [ ]:
length(list_of_files)

In [ ]:
mutable struct VAE{E, D}
    encoder::E
    latent::Flux.Parallel
    decoder::D

end

# fixed architecture for now
function VAE(input_dim, latent_dim, output_dim; encoder_hidden_dims = [128, 80], decoder_hidden_dims = [80, 240, 680,], activation = relu)

    encoder = GNN.GNNChain(
        GNN.GCNConv(input_dim => encoder_hidden_dims[1] , activation),
        Flux.LayerNorm(encoder_hidden_dims[1]),
        GNN.GCNConv(encoder_hidden_dims[1] => encoder_hidden_dims[2], activation),
        Flux.LayerNorm(encoder_hidden_dims[2]),
        GNN.GlobalPool(Statistics.mean),
    )

    latent = Flux.Parallel(
        tuple, 
        Flux.Dense(encoder_hidden_dims[2], latent_dim, identity), # Mean, no activation to not restrict expressivity of mean 
        Flux.Dense(encoder_hidden_dims[2], latent_dim, identity)  # Log variance, no activation to not restrict expressivity of log(sigma) 
    )

    decoder = Flux.Chain(
        Flux.Dense(latent_dim, decoder_hidden_dims[1], activation),
        Flux.LayerNorm(decoder_hidden_dims[1]),
        Flux.Dense(decoder_hidden_dims[1], decoder_hidden_dims[2], activation),
        Flux.LayerNorm(decoder_hidden_dims[2]),
        Flux.Dense(decoder_hidden_dims[2], decoder_hidden_dims[3], activation),
        Flux.LayerNorm(decoder_hidden_dims[3]),
        Flux.Dense(decoder_hidden_dims[3], output_dim, Flux.sigmoid) # Output reconstruction
    )

    return VAE(encoder, latent, decoder)

end

Flux.@layer VAE

# reparameterization trick to be able to differentiate through the sampling process
function reparameterize(µ::AbstractArray, logσ²::AbstractArray)::AbstractArray
    # Sample from the standard normal distribution
    ε = randn(size(µ))|> Flux.gpu
    
    # Reparameterization trick
    z = µ .+ exp.(0.5f0 .* logσ²) .* ε

    return z
end

# Forward pass through the VAE
function (vae::VAE)(g::GNN.GNNGraph, x::AbstractArray)
    # Encode input to get latent distribution parameters
    (µ, logσ²) = vae.encoder(g, x) |> vae.latent
    
    # Sample z from the latent distribution
    z = reparameterize(µ, logσ²)
    
    # Decode z to get reconstruction
    x̂ = vae.decoder(z)
    
    return x̂, µ, logσ²
end


# VAE loss function: reconstruction loss + KL divergence
function elbo_vae_loss(g::GNN.GNNGraph, model::VAE; beta::Float32 = 1.0f0)
    # Get the true adjacency matrix from the graph
    adj_true = GNN.adjacency_matrix(g) # This is the ground truth adjacency matrix
    
    # Flatten it to match decoder output format
    adj_true_flat = vec(adj_true)
    
    # Forward pass through model - x is just node features, not used in loss
    adj_pred_flat, μ, logσ² = out = vae(g, reshape(g.ndata.x, (:, 1)))

    # Reconstruction loss
    rec_loss = Flux.logitbinarycrossentropy(adj_true_flat, adj_pred_flat, agg=sum) / size(x, 2)

    #  giving dim=2 here is not necessary anymore because we only have one dimension left
    kl_div = 0.5f0 .* Statistics.mean(sum(exp.(logσ²) .+  µ .* µ .- 1f0 .- logσ², dims=1))
    return rec_loss + beta*kl_div, rec_loss, kl_div
end


In [ ]:
function encode_data(n_max, n_nodes, linkMatrix)::GNN.GNNGraph    
    # we can construct other node features here if we need them, like we did before or in another way
    # here we just us ones because the damn package doesn't work with empty node features

    # more meaningful node features: 
    
    # 1. Causal layer (approximate using in/out degree)
    # in_degree = vec(sum(linkMatrix, dims=1))
    # out_degree = vec(sum(linkMatrix, dims=2))
    
    # # 2. Temporal position estimation
    # # In a DAG, this can approximate position in causal ordering
    # temporal = zeros(Float32, n_max)
    # for i in 1:n_nodes
    #     temporal[i] = out_degree[i] / (in_degree[i] + 1)  # Add 1 to avoid division by zero
    # end

    return GNN.GNNGraph(linkMatrix; ndata = vcat(ones(Float32, n_nodes), zeros(Float32, n_max - n_nodes)))
end

In [ ]:

# VAE loss function: reconstruction loss + KL divergence
function elbo_vae_loss(g::GNN.GNNGraph, model::VAE; beta::Float32 = 1.0f0)
    # Get the true adjacency matrix from the graph
    adj_true = GNN.adjacency_matrix(g) # This is the ground truth adjacency matrix
    
    # Flatten it to match decoder output format
    adj_true_flat = vec(adj_true)
    
    # Forward pass through model - x is just node features, not used in loss
    adj_pred_flat, μ, logσ² = model(g, reshape(g.ndata.x, (:, 1)))

    # Reconstruction loss
    rec_loss = Flux.logitbinarycrossentropy(adj_true_flat, adj_pred_flat, agg=sum) / size(g.ndata.x, 2)

    #  giving dim=2 here is not necessary anymore because we only have one dimension left
    kl_div = 0.5f0 .* Statistics.mean(sum(exp.(logσ²) .+  µ .* µ .- 1f0 .- logσ², dims=1))
    return rec_loss + beta*kl_div, rec_loss, kl_div
end

function elbo_vae_loss_batched(gs::AbstractArray, mode::VAE; beta::Float32 = 1.0f0)
    total = 0.0f0
    rec_loss = 0.0f0
    kl_div = 0.0f0
    for g in gs 
        loss, rec, kl = elbo_vae_loss(g, mode; beta=beta)
        total += loss
        rec_loss += rec
        kl_div += kl
    end
    # mean loss over batch to normalize away batch size dependency
    return total/length(gs), rec_loss/length(gs), kl_div/length(gs)
end

In [ ]:
# hyper parameters
input_dim = 110 
latent_dim = 36
output_dim = input_dim*input_dim
encoder_hidden_dims = [128, 80]
decoder_hidden_dims = [80, 240, 680]
activation = Flux.relu # I am pretty sure this isn't right I need to use tanh or sigmoid here
batchsize = 128
epochs = 1000
beta = 1.0f0
learning_rate = 1e-3
# for early stopping
current_loss = Inf
last_loss = Inf
patience = 15 

# model
device = CUDA.functional() ? Flux.gpu : Flux.cpu 

vae = VAE(input_dim, latent_dim, output_dim; encoder_hidden_dims = encoder_hidden_dims, decoder_hidden_dims = decoder_hidden_dims, activation = activation) |> device

# dataset
dset = QG.DataLoader.Dataset(
    path_to_data, 
    Random.shuffle(StatsBase.sample(list_of_files, Int(ceil(length(list_of_files)/2)))),
    cache_size = 5,
    columns = [ :linkMatrix, :n, :nmax],
)

graphs = [encode_data(Int(dset[i].nmax), Int(dset[i].n), dset[i].linkMatrix) for i in 1:length(dset)]

partition = Flux.splitobs( graphs, at=(0.7, 0.15, 0.15))[1:3]

train_loader = Flux.DataLoader(
    partition[1],
    shuffle = true , 
    batchsize = batchsize,
    collate = true, 
    parallel = true
)

val_loader = Flux.DataLoader(
    partition[2],
    shuffle = true , 
    batchsize = batchsize,
    collate = true, 
    parallel = true
)
   
test_loader = Flux.DataLoader(
    partition[3],
    shuffle = true , 
    batchsize = batchsize,
    collate = true, 
    parallel = true
)

# optimizer
opt = Flux.setup(Flux.Adam(learning_rate), vae)

v_loss_total = []
train_loss_total = []
v_loss_rec = []
train_loss_rec = []
v_loss_kl = []
train_loss_kl = []

# training loop
for epoch in 1:epochs 
    println("Epoch: $epoch")
    all_loss_sum= 0.0f0
    rec_loss_sum = 0.0f0
    kl_loss_sum = 0.0f0
    @showprogress for (i, batch) in enumerate(train_loader) 
        # println("Batch: $(i)/$(length(train_loader))")
        graphs = MLUtils.unbatch(batch) |> device

        (all_loss, rec_loss, kl_loss,), (grads,) = Flux.withgradient(vae) do model
            elbo_vae_loss_batched(graphs, model; beta=beta)
        end

        # Check for numerical stability issues
        if any(isnan, all_loss)
            @warn "NaN detected in loss at epoch $epoch"
            hasnan = true
            break
        end
        all_loss_sum += all_loss
        rec_loss_sum += rec_loss
        kl_loss_sum += kl_loss
        # Update parameters
        Flux.update!(opt, vae, grads)
    end

    push!(train_loss_total, all_loss_sum/length(train_loader))
    push!(train_loss_rec, rec_loss_sum/length(train_loader))
    push!(train_loss_kl, kl_loss_sum/length(train_loader))
    

    v_l_total = 0.0f0
    v_l_rec = 0.0f0
    v_l_kl = 0.0f0
    for (i, batch) in enumerate(valid_loader)
        gbatch = [encode_data(n_max, Int(element.n), element.linkMatrix) for element in batch] |> device

        # Compute loss
        all_loss, rec_loss, kl_div = elbo_vae_loss_batched(gbatch, vae; beta=beta)
        validation_loss[i] = all_loss

        # Check for numerical stability issues
        if any(isnan, all_loss)
            @warn "NaN detected in loss at epoch $epoch"
            hasnan = true
            break
        end
        v_l_total += all_loss
        v_l_rec += rec_loss
        v_l_kl += kl_div
    end
    push!(v_loss_total, v_l_total/length(valid_loader))
    push!(v_loss_rec, v_l_rec/length(valid_loader))
    push!(v_loss_kl, v_l_kl/length(valid_loader))

    @info "Epoch $epoch: Training Loss: $(train_loss_total[end]), Validation Loss: $(v_loss_total[end], "patience: $patience")"

    current_loss = v_loss_total[end]

    if current_loss < last_loss*0.99
        patience = 15
        last_loss = current_loss
    else
        patience -= 1
        if patience == 0
            @info "Early stopping at epoch $epoch"
            break 
        end
    end
end